In [0]:
#load csv load
df = spark.read.csv(
    "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv",
    header=True,
    inferSchema=True
)


In [0]:
#create delta path 
delta_path = "dbfs:/FileStore/ecommerce/delta/events"



In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("ecommerce_events")


In [0]:
spark.table("ecommerce_events").count()


67501979

In [0]:
spark.table("ecommerce_events").printSchema()
spark.table("ecommerce_events").count()



root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)



67501979

In [0]:
spark.sql('DESCRIBE DETAIL ecommerce_events')


DataFrame[format: string, id: string, name: string, description: string, location: string, createdAt: timestamp, lastModified: timestamp, partitionColumns: array<string>, clusteringColumns: array<string>, numFiles: bigint, sizeInBytes: bigint, properties: map<string,string>, minReaderVersion: int, minWriterVersion: int, tableFeatures: array<string>, statistics: map<string,bigint>, clusterByAuto: boolean]

In [0]:
#Create bad data (wrong data type)
from pyspark.sql import Row

bad_df = spark.createDataFrame([
    Row(event_type="purchase", price="FREE", user_id=12345)
])


In [0]:
bad_df.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable("ecommerce_events1")


In [0]:
#Insert duplicates, detect them, then fix them properly
dup_df = spark.table("ecommerce_events1").limit(1000)


In [0]:
dup_df.write.format("delta").mode("append").saveAsTable("ecommerce_events1")
dup_df.write.format("delta").mode("append").saveAsTable("ecommerce_events1")


In [0]:
df1 = spark.sql("""
          SELECT user_id, COUNT(*) AS cnt FROM ecommerce_events1
GROUP BY user_id
HAVING cnt > 1;
""")

In [0]:
df1.show(3)

+-------+---+
|user_id|cnt|
+-------+---+
|  12345|  4|
+-------+---+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy(
    "user_id", "product_id", "event_time"
).orderBy("event_time")

dedup_df = spark.table("ecommerce_events") \
    .withColumn("rn", row_number().over(window_spec)) \
    .filter("rn = 1") \
    .drop("rn")


In [0]:
dedup_df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("ecommerce_events1")
